<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания
Вариант № 11 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Customer в C#, который будет представлять информацию о 
клиентах или покупателях. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [13]:
abstract class Customer
{
    private int _customerId;
    private string _name;
    private string _email;

    public int CustomerId
    {
        get => _customerId;
        protected set
        {
            if (value <= 0) throw new ArgumentException("CustomerId должен быть > 0");
            _customerId = value;
        }
    }

    public string Name
    {
        get => _name;
        protected set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Имя не может быть пустым");
            _name = value.Trim();
        }
    }

    public string Email
    {
        get => _email;
        protected set
        {
            // Простая валидация для примера (без Regex)
            if (string.IsNullOrWhiteSpace(value) || !value.Contains("@") || value.StartsWith("@") || value.EndsWith("@"))
                throw new ArgumentException("Некорректный email");
            _email = value.Trim();
        }
    }

    protected Customer(int customerId, string name, string email)
    {
        CustomerId = customerId;
        Name = name;
        Email = email;
    }

    public virtual string GetFullName()
    {
        return Name;
    }

    public virtual void UpdateEmail(string newEmail)
    {
        Email = newEmail;
        Console.WriteLine($"Почта пользователя ID:{CustomerId} {Name} изменена");
    }

    public virtual void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId}, Имя: {Name}, Email: {Email}");
    }

    public virtual void SendEmail(Customer to, string subject, string body)
    {
        if (to is null) throw new ArgumentNullException(nameof(to));
        Console.WriteLine(
            $"[EMAIL]\n  От: {GetFullName()} <{Email}>\n  Куда: {to.GetFullName()} <{to.Email}>\n  Тема: {subject}\n  Сообщение: {body}\n");
    }
}

class VipCustomer : Customer
{
    private int _loyaltyPoints;

    public int LoyaltyPoints
    {
        get => _loyaltyPoints;
        private set
        {
            if (value < 0) throw new ArgumentException("Баллы лояльности не могут быть отрицательными");
            _loyaltyPoints = value;
        }
    }

    public VipCustomer(int customerId, string name, string email, int loyaltyPoints = 0)
        : base(customerId, name, email)
    {
        LoyaltyPoints = loyaltyPoints;
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId}, Роль: Вип пользователь, Имя: {Name}, Email: {Email}, Баланс Лояльности: {LoyaltyPoints}");
    }

    public void TransferPoints(VipCustomer to, int amount)
    {
        if (to is null) throw new ArgumentNullException(nameof(to));
        if (amount <= 0) throw new ArgumentException("Сумма перевода должна быть > 0");
        if (LoyaltyPoints < amount) throw new InvalidOperationException("Недостаточно баллов для перевода");

        LoyaltyPoints -= amount;
        to.LoyaltyPoints += amount;

        Console.WriteLine($"Перевод баллов: {GetFullName()} → {to.GetFullName()} : {amount} (остаток у отправителя: {LoyaltyPoints})");
    }

    public void AddPurchaseBonus(decimal purchaseAmount)
    {
        int bonus = (int)Math.Max(1, Math.Floor(purchaseAmount / 10m)); // 1 балл за каждые ~10.
        LoyaltyPoints += bonus;
        Console.WriteLine($"{GetFullName()} получил {bonus} балл(ов) за покупку на сумму {purchaseAmount}. Итого: {LoyaltyPoints}");
    }
}

class RegularCustomer : Customer
{
    private DateTime _registrationDate;

    public DateTime RegistrationDate
    {
        get => _registrationDate;
        private set
        {
            if (value == default) throw new ArgumentException("Дата регистрации не задана");
            _registrationDate = value;
        }
    }

    public DateTime LastEmailUpdate { get; private set; }

    public RegularCustomer(int customerId, string name, string email, DateTime registrationDate)
        : base(customerId, name, email)
    {
        RegistrationDate = registrationDate;
        LastEmailUpdate = registrationDate;
    }

    public override void UpdateEmail(string newEmail)
    {
        base.UpdateEmail(newEmail);
        LastEmailUpdate = DateTime.Now;
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId}, Роль: Обычный пользователь, Имя: {Name}, Email: {Email}, Зарегистрирован: {RegistrationDate}, Последнее обновление email: {LastEmailUpdate}");
    }

    public void JoinGroup(GroupCustomer group)
    {
        if (group is null) throw new ArgumentNullException(nameof(group));
        group.AddMember(this);
    }
}

class GroupCustomer : Customer
{
    private string _groupName;

    public string GroupName
    {
        get => _groupName;
        private set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название группы не может быть пустым");
            _groupName = value.Trim();
        }
    }

    // Состав группы
    private readonly HashSet<int> _memberIds = new HashSet<int>();
    public IReadOnlyCollection<int> Members => _memberIds;

    public GroupCustomer(int customerId, string groupName, string email)
        : base(customerId, groupName, email)
    {
        GroupName = groupName;
    }

    public override string GetFullName()
    {
        return GroupName;
    }

    public override void ViewProfile()
    {
        Console.WriteLine($"ID: {CustomerId}, Группа: {GroupName}, Email: {Email}");
    }

    public void AddMember(Customer member)
    {
        if (member is null) throw new ArgumentNullException(nameof(member));
        if (_memberIds.Add(member.CustomerId))
        {
            Console.WriteLine($"[{GroupName}] Добавлен участник: {member.GetFullName()} (ID={member.CustomerId})");
        }
        else
        {
            Console.WriteLine($"[{GroupName}] Участник уже в группе: {member.GetFullName()} (ID={member.CustomerId})");
        }
    }

    public void RemoveMember(Customer member)
    {
        if (member is null) throw new ArgumentNullException(nameof(member));
        if (_memberIds.Remove(member.CustomerId))
        {
            Console.WriteLine($"[{GroupName}] Удалён участник: {member.GetFullName()} (ID={member.CustomerId})");
        }
        else
        {
            Console.WriteLine($"[{GroupName}] Такого участника нет: {member.GetFullName()} (ID={member.CustomerId})");
        }
    }

    public void Broadcast(string subject, string body)
    {
        Console.WriteLine($"[{GroupName}] Рассылка по группе ({_memberIds.Count} получателей).\n  Тема: {subject}\n  {body}\n");
    }
}


var vip1 = new VipCustomer(1, "Никита", "nikita@mail.com", loyaltyPoints: 120);
var vip2 = new VipCustomer(4, "Анна", "anna@mail.com", loyaltyPoints: 50);

var regular = new RegularCustomer(2, "Егор", "egor@mail.com", new DateTime(2022, 5, 1));
var group = new GroupCustomer(3, "Студенты-22Б", "group@mail.com");

// Просмотр профилей (полиморфизм через override ViewProfile)
Customer[] customers = { vip1, vip2, regular, group };
Console.WriteLine("=====Профили всех созданных пользователей:=====");
foreach (var c in customers) c.ViewProfile();

Console.WriteLine();

// Взаимодействие: обновление email + отметка времени у RegularCustomer
Console.WriteLine("=====Тест изменения почты=====:");
regular.UpdateEmail("egor.new@mail.com");
regular.ViewProfile();

Console.WriteLine();

// Взаимодействие: вступление в группу и рассылка
Console.WriteLine("=====Тест вступления в группы=====");
regular.JoinGroup(group);
group.AddMember(vip1); // добавим и VIP
group.ViewProfile();
Console.WriteLine();
group.Broadcast("Старт семестра", "У нас первая встреча в пятницу в 12:00 в аудитории 302.");

Console.WriteLine();

// Взаимодействие: отправка писем между клиентами
Console.WriteLine("=====Тест отправки почты=====");
vip1.SendEmail(regular, "Привет", "Как успехи?");
regular.SendEmail(vip1, "Re: Привет", "Всё ок, готовлюсь к занятию.");
group.SendEmail(vip2, "Приглашение", "Присоединяйтесь к нашей группе!");

Console.WriteLine();

// Взаимодействие: перевод баллов между VIP + начисление бонуса
Console.WriteLine("=====Тест перевода бонусов=====");
vip1.TransferPoints(vip2, 30);
vip2.AddPurchaseBonus(135.60m);

Console.WriteLine();

// Финальный вывод
Console.WriteLine("=====Профили всех созданных пользователей:=====");
foreach (var c in customers) c.ViewProfile();



=====Профили всех созданных пользователей:=====
ID: 1, Роль: Вип пользователь, Имя: Никита, Email: nikita@mail.com, Баланс Лояльности: 120
ID: 4, Роль: Вип пользователь, Имя: Анна, Email: anna@mail.com, Баланс Лояльности: 50
ID: 2, Роль: Обычный пользователь, Имя: Егор, Email: egor@mail.com, Зарегистрирован: 5/1/2022 12:00:00 AM, Последнее обновление email: 5/1/2022 12:00:00 AM
ID: 3, Группа: Студенты-22Б, Email: group@mail.com

=====Тест изменения почты=====:
Почта пользователя ID:2 Егор изменена
ID: 2, Роль: Обычный пользователь, Имя: Егор, Email: egor.new@mail.com, Зарегистрирован: 5/1/2022 12:00:00 AM, Последнее обновление email: 9/21/2025 11:35:07 PM

=====Тест вступления в группы=====
[Студенты-22Б] Добавлен участник: Егор (ID=2)
[Студенты-22Б] Добавлен участник: Никита (ID=1)
ID: 3, Группа: Студенты-22Б, Email: group@mail.com

[Студенты-22Б] Рассылка по группе (2 получателей).
  Тема: Старт семестра
  У нас первая встреча в пятницу в 12:00 в аудитории 302.


=====Тест отправки п